# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yashkrverma1234-glitch/ml-internship-assignment1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Logistic Regression, then Random Forest** — the same progression the toolkit recommends for
"yes/no with an observed label." ML-03 already framed this as scoring/ranking with a binary
classification sub-task underneath (`is_declining_label`), and its feature-importance check found
"no single feature dominates" (top feature only 15.8% share) — a real but tangled pattern, which is
exactly the case for trying a model past a hand-written rule. Logistic regression comes first because
it's readable (a coefficient per feature, sign and magnitude both mean something); Random Forest comes
second to see whether the extra flexibility actually earns its keep on this feature set. Both are
compared against the *same* Week-4 baseline rule, on the *same* rows, at the *same* metric
(precision@K), so any lift is attributable to the model — not to new information or a different test set.


In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/yashkrverma1234-glitch/ml-internship-assignment1"
REPO_DIR = "ml-internship-assignment1"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

os.makedirs("work/outputs", exist_ok=True)
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"

import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# Same baseline rule as work/notebooks/w04_baseline_score.ipynb, so the comparison
# in Section 3 is apples-to-apples: same data, same rows, same score formula.
tier_median_ctr = df.loc[df["avg_position"] > 0].groupby("position_tier")["ctr"].median()
df["tier_median_ctr"] = df["position_tier"].map(tier_median_ctr)
VOLUME_FLOOR = 500
eligible = (df["avg_position"] > 0) & (df["impressions_90d"] >= VOLUME_FLOOR) & df["tier_median_ctr"].notna()
stale_bonus = np.where(df["days_since_last_update"] >= 180, 1.15, 1.0)
df["ctr_gap"] = np.where(eligible, (df["tier_median_ctr"] - df["ctr"]).clip(lower=0), 0.0)
df["baseline_action_score"] = np.where(eligible, df["ctr_gap"] * np.log1p(df["impressions_90d"]) * stale_bonus, 0.0)

# Feature vector: the same signals Week-3/4 already vetted for leakage, minus
# trend_direction/trend_pct themselves (those ARE the label, so they're excluded).
NUMERIC_FEATURES = ["impressions_90d", "clicks_90d", "sessions_90d", "ctr", "avg_position",
                     "content_age_days", "days_since_last_update", "word_count"]
CATEGORICAL_FEATURES = [c for c in ["content_type", "position_tier", "main_intent"] if c in df.columns]

num = df[NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").fillna(0)
num["log_impressions_90d"] = np.log1p(num["impressions_90d"])
num["log_clicks_90d"] = np.log1p(num["clicks_90d"])
num["log_sessions_90d"] = np.log1p(num["sessions_90d"])
cat = df[CATEGORICAL_FEATURES].fillna("unknown").astype(str)
cat_dummies = pd.get_dummies(cat, prefix=CATEGORICAL_FEATURES, dtype=float)

X = pd.concat([num.reset_index(drop=True), cat_dummies.reset_index(drop=True)], axis=1)
y = df["is_declining_label"].astype(int)

print(f"Rows: {len(df):,}  |  Features: {X.shape[1]}  |  Overall decline rate: {y.mean():.3f}")
print(f"Excluded from features (label-source): trend_direction, trend_pct")
print(f"Excluded from features (identifier, not a signal): content_id, client_id")


Rows: 30,000  |  Features: 24  |  Overall decline rate: 0.542
Excluded from features (label-source): trend_direction, trend_pct
Excluded from features (identifier, not a signal): content_id, client_id

## 2. Split design

**Client-grouped holdout — 20% of clients (by count), entirely held out, `random_state=42`.**
Grouped by `client_id`, not a random row split. With only 32 pseudonymized clients and multiple
pages per client, a random row split would let some of the *same* client's other pages leak into
training — the model could then partly win by memorizing a client's template or industry quirks
instead of learning a signal that transfers to a page from a client it has never scored before.
That's the actual decision this lane serves (rank *any* client's queue), so the split has to match it.

One honest catch, visible in the code output below: with only 6 clients in the test fold, their
per-client decline rates range from 0% (n=3, thin) to 61.9% — so the precision@K numbers in Section 3
are somewhat sensitive to *which* clients this particular `random_state` happened to hold out, not
purely to model quality. That instability is exactly what ML-09's validation audit exists to
stress-test (e.g. repeating this split under several seeds).


In [2]:
RANDOM_STATE = 42

clients = df["client_id"].astype(str)
unique_clients = clients.drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
shuffled = rng.permutation(unique_clients)
n_test_clients = max(1, round(len(shuffled) * 0.2))
test_clients = set(shuffled[:n_test_clients])
test_mask = clients.isin(test_clients).to_numpy()

train_idx = np.where(~test_mask)[0]
test_idx = np.where(test_mask)[0]

print(f"Split: client-grouped holdout, {len(test_clients)} of {len(unique_clients)} clients held out entirely (random_state={RANDOM_STATE})")
print(f"Train: {len(train_idx):,} rows across {len(unique_clients) - len(test_clients)} clients (decline rate {y.iloc[train_idx].mean():.3f})")
print(f"Test:  {len(test_idx):,} rows across {len(test_clients)} clients  (decline rate {y.iloc[test_idx].mean():.3f})")
print()
print("Why grouped, not a random row split: a random split would put some of the SAME client's")
print("pages in both train and test. The model could then partly succeed by memorizing a client's")
print("template/industry quirks instead of learning a pattern that transfers to a page from a client")
print("it has never seen -- which is the real decision this needs to support (any client's queue).")
print()
print("Per-client decline rate in the held-out test clients (shows why the test score can be noisy):")
print(df.loc[test_mask].groupby("client_id")["is_declining_label"].agg(n="size", decline_rate="mean").sort_values("decline_rate"))


Split: client-grouped holdout, 6 of 32 clients held out entirely (random_state=42)
Train: 27,675 rows across 26 clients (decline rate 0.555)
Test:  2,325 rows across 6 clients  (decline rate 0.391)

Why grouped, not a random row split: a random split would put some of the SAME client's
pages in both train and test. The model could then partly succeed by memorizing a client's
template/industry quirks instead of learning a pattern that transfers to a page from a client
it has never seen -- which is the real decision this needs to support (any client's queue).

Per-client decline rate in the held-out test clients (shows why the test score can be noisy):
                      n  decline_rate
client_id                            
client_1a6562590e     3      0.000000
client_d4735e3a26  1106      0.219711
client_0b918943df    35      0.485714
client_4fc82b26ae    32      0.531250
client_f74efabef1  1031      0.542192
client_98a3ab7c34   118      0.618644

## 3. Train + compare vs my baseline

**Same data** (`data/raw/content_refresh_anonymized.csv`), **same baseline score** (Week-4's
CTR-gap-vs-position-tier rule, recomputed identically), **same split and target** as above
(client-grouped holdout, `is_declining_label`). Table from the code below:

| | precision@20 | precision@50 | precision@100 | ROC AUC | avg precision |
|---|---:|---:|---:|---:|---:|
| baseline rule (Week-4 CTR-gap score) | 0.80 | 0.72 | 0.67 | 0.543 | 0.437 |
| logistic_regression | 0.80 | **0.74** | **0.74** | 0.725 | **0.592** |
| random_forest | 0.50 | 0.62 | 0.63 | **0.737** | 0.580 |

**The honest finding, not the tidy one:** Random Forest does *not* win here, despite being the
"stronger" model on paper — Logistic Regression beats it at precision@20/50/100 and average
precision; Random Forest only edges ahead on raw ROC AUC. The baseline rule is competitive at the
very top of the queue (precision@20 ties Logistic Regression at 0.80) but its ROC AUC (0.543) is
barely above chance — its top picks happen to be right more than its overall ranking is good,
because it only ever scores a minority of eligible pages (the ones clearing the volume floor) and
leaves everything else at zero. Logistic Regression is the model I'd actually ship for this queue:
it clearly beats the baseline everywhere that matters (deeper into the queue, not just the top 20)
and it stays interpretable.


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score
import json

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
baseline_test = df.iloc[test_idx]["baseline_action_score"].to_numpy()

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y_true)[order].mean())

def score_row(y_true, scores):
    return {
        "precision_at_20": precision_at_k(y_true, scores, 20),
        "precision_at_50": precision_at_k(y_true, scores, 50),
        "precision_at_100": precision_at_k(y_true, scores, 100),
        "roc_auc": float(roc_auc_score(y_true, scores)),
        "average_precision": float(average_precision_score(y_true, scores)),
    }

results = {"baseline_rule (Week-4 CTR-gap score)": score_row(y_test, baseline_test)}

logreg = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
])
logreg.fit(X_train, y_train)
proba_lr = logreg.predict_proba(X_test)[:, 1]
results["logistic_regression"] = score_row(y_test, proba_lr)

rf = RandomForestClassifier(class_weight="balanced_subsample", n_estimators=300, max_depth=8,
                             min_samples_leaf=20, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)
proba_rf = rf.predict_proba(X_test)[:, 1]
results["random_forest"] = score_row(y_test, proba_rf)

comparison = pd.DataFrame(results).T[["precision_at_20", "precision_at_50", "precision_at_100", "roc_auc", "average_precision"]]
print(comparison.round(3).to_string())

best_by_p50 = comparison["precision_at_50"].idxmax()
print(f"\nBest at precision@50: {best_by_p50}")
print("Note: random_forest does NOT win here despite being the 'stronger' model on paper --")
print("logistic_regression beats it at every K on this split. Reporting the loss, not hiding it.")

with open("work/outputs/w05_model_results.json", "w") as f:
    json.dump({
        "target": "is_declining_label",
        "split_strategy": "client_grouped_holdout",
        "random_state": RANDOM_STATE,
        "train_rows": int(len(train_idx)), "test_rows": int(len(test_idx)),
        "test_clients_held_out": int(len(test_clients)),
        "metrics": results,
        "best_model_by_precision_at_50": best_by_p50,
    }, f, indent=2)
print("\nWrote work/outputs/w05_model_results.json")


                                      precision_at_20  precision_at_50  precision_at_100  roc_auc  average_precision
baseline_rule (Week-4 CTR-gap score)              0.8             0.72              0.67    0.543              0.437
logistic_regression                               0.8             0.74              0.74    0.725              0.592
random_forest                                     0.5             0.62              0.63    0.737              0.580

Best at precision@50: logistic_regression
Note: random_forest does NOT win here despite being the 'stronger' model on paper --
logistic_regression beats it at every K on this split. Reporting the loss, not hiding it.

Wrote work/outputs/w05_model_results.json

## 4. Errors and interpretation

**What it leans on:** for both models, `log_impressions_90d` is the single strongest input (RF
importance 0.164; the largest-magnitude logistic coefficient, positive). Plausibility check: more
impressions is *not* obviously "should be declining," so this deserves a sanity note rather than
blind trust — with a 54% overall decline rate, higher-traffic pages are also more exposed to SERP
volatility (more of them get flagged as some kind of "down" in a snapshot with this much churn), which
is a plausible mechanism, not proof. It is not "suspiciously perfect" (no single feature clears even
20% importance), so I don't read this as leakage — but it is the first thing I'd re-check against a
second time window in ML-09.

**Where it's wrong, concretely (from the code below):**
- *False negatives* (actually declining, scored lowest): all three are extremely low-traffic pages
  (1–5 impressions/90d) from `client_d4735e3a26` — the client with the lowest test-set decline rate
  (22%). The model reads "almost no traffic" as "nothing much is happening" rather than "declining
  toward zero," and it may also be under-weighting this specific, atypical client.
- *False positives* (labeled "stable", scored highest): all three are the opposite profile —
  high-traffic (2.8k–8.8k impressions/90d), recently updated (20 days), mid-position (11–14) pages
  from `client_f74efabef1`. The model has learned "high traffic + mid position" as a decline signal
  elsewhere in the data, and over-applies it here even though these particular pages are holding
  steady.

**Takeaway for the reviewer this feeds:** trust the model most for pages with real, established
traffic; treat both very-low-traffic pages and any single client's queue with extra manual review,
since both are where the errors cluster.


In [4]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
coefs = pd.Series(logreg.named_steps["model"].coef_[0], index=X.columns).sort_values(key=abs, ascending=False)

print("Top 8 random forest feature importances:")
print(importances.head(8).round(3).to_string())
print("\nTop 8 |logistic regression coefficients| (standardized features, sign shows direction):")
print(coefs.head(8).round(3).to_string())

test_df = df.iloc[test_idx].copy().reset_index(drop=True)
test_df["y_true"] = y_test.to_numpy()
test_df["proba_lr"] = proba_lr
cols = ["content_id", "client_id", "proba_lr", "impressions_90d", "content_age_days",
        "days_since_last_update", "avg_position", "trend_direction"]

fn = test_df[test_df.y_true == 1].sort_values("proba_lr").head(3)
fp = test_df[test_df.y_true == 0].sort_values("proba_lr", ascending=False).head(3)

print("\nFalse negatives (actually declining, model scored them lowest):")
print(fn[cols].round(3).to_string(index=False))
print("\nFalse positives (not declining, model scored them highest):")
print(fp[cols].round(3).to_string(index=False))


Top 8 random forest feature importances:
log_impressions_90d       0.164
content_age_days          0.160
impressions_90d           0.156
avg_position              0.138
word_count                0.075
log_clicks_90d            0.040
days_since_last_update    0.040
position_tier_top_3       0.035

Top 8 |logistic regression coefficients| (standardized features, sign shows direction):
log_impressions_90d       0.983
log_clicks_90d           -0.686
avg_position              -0.352
content_age_days          -0.338
log_sessions_90d          -0.320
position_tier_top_3       -0.214
word_count                 0.172
days_since_last_update     0.108

False negatives (actually declining, model scored them lowest):
          content_id         client_id  proba_lr  impressions_90d  content_age_days  days_since_last_update  avg_position trend_direction
content_a8cee66e4788 client_d4735e3a26     0.021                1               489                      20           2.0            down
content_867

## Self-check

Before you submit, confirm each line honestly:

- [yes] Every section above is filled — markdown thinking AND the code that backs it
- [yes] The notebook runs top to bottom with no errors (Runtime → Run all)
- [yes] No client names, URLs, or private queries anywhere
- [yes] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
